# Currency & Economy

Currency flow simulation for Animal Company. Models instance value-in/out, wallet balances, Key Card progression, and Battle Pass economics.

> Reference: `animalco_economy.md` in brain-rag vault

In [1]:
from IPython.display import HTML
display(HTML('''
<style id="aco-toggle-style"></style>
<button id="aco-toggle-btn"
  onclick="
    var style = document.getElementById('aco-toggle-style');
    if (style.innerHTML === '') {
      style.innerHTML = '.jp-Cell-inputWrapper { display: none !important; }';
      this.innerHTML = 'Show Code';
    } else {
      style.innerHTML = '';
      this.innerHTML = 'Hide Code';
    }
  "
  style="padding: 6px 16px; font-size: 13px; cursor: pointer;
         border: 1px solid #ccc; border-radius: 4px; background: #f5f5f5;">
  Hide Code
</button>
'''))

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)

from aco_model.models import EconomyParams, RetentionCurve
from aco_model.retention import load_installs, simulate
from aco_model.economy import simulate_economy, simulate_player_progression
from aco_model.config import load_config
from aco_model.state import load_state, save_state, DEFAULT_STATE_PATH

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

_HEADLESS = os.environ.get('ACO_HEADLESS') == '1'

cfg = load_config()
state = load_state()

if state is not None:
    installs = load_installs(cfg.installs_path)
    curve = RetentionCurve(anchors=state.retention_anchors)
    sim = simulate(installs, curve, state.sim_days)
    econ_params = state.economy or cfg.economy
    print(f'=== Loaded from shared state ===')
else:
    installs = load_installs(cfg.installs_path)
    sim = simulate(installs, cfg.retention, cfg.sim_days)
    econ_params = cfg.economy
    print(f'=== Using config.yaml defaults ===')

# Load extra economy values from raw config (not in Pydantic model)
import yaml as _yaml
with open('config.yaml') as _f:
    _raw_cfg = _yaml.safe_load(_f) or {}
_econ_raw = _raw_cfg.get('economy', {})
_saved_bronze_cost = _econ_raw.get('bronze_coin_cost', 10.00)
_saved_nut_usd = _econ_raw.get('nut_value_usd', 0.01)
_saved_scrap_usd = _econ_raw.get('scrap_value_usd', 0.005)

print(f'  DAU range: {sim.dau.min():,} - {sim.dau.max():,}')
print(f'  Instances/day: {econ_params.instances_per_day}')
print(f'  Sim days: {sim.sim_days}')


=== Loaded from shared state ===
  DAU range: 15,182 – 61,725
  Instances/day: 1


## 1. Resource Values & Exchange Rates

In [ ]:
# ── Resource value inputs ─────────────────────────────────────────────────
coin_value = widgets.FloatText(value=econ_params.coin_to_usd, description='1 Coin = $', step=0.1,
                                layout=widgets.Layout(width='180px'))
nut_value = widgets.FloatText(value=_saved_nut_usd, description='1 Nut = $', step=0.001,
                               layout=widgets.Layout(width='180px'))
scrap_value = widgets.FloatText(value=_saved_scrap_usd, description='1 Scrap = $', step=0.001,
                                 layout=widgets.Layout(width='180px'))

# ── Seeded currency inputs ────────────────────────────────────────────────
seed_coins = widgets.FloatText(value=econ_params.seed_coins, description='Seed Coins:', step=1,
                                layout=widgets.Layout(width='180px'))
seed_nuts = widgets.FloatText(value=econ_params.seed_nuts, description='Seed Nuts:', step=100,
                               layout=widgets.Layout(width='180px'))
seed_scrap = widgets.FloatText(value=econ_params.seed_scrap, description='Seed Scrap:', step=100,
                                layout=widgets.Layout(width='180px'))

# ── Key Card tier inputs ──────────────────────────────────────────────────
_kc_tiers = econ_params.keycard_tiers
_kc_names = [kc.name for kc in _kc_tiers]

_w = widgets.Layout(width='120px')
bronze_coin_cost = widgets.FloatText(value=_saved_bronze_cost, step=0.01, layout=_w)

_kc_derived_prices = {}
_kc_usd_labels = {}
for kc in _kc_tiers:
    _kc_derived_prices[kc.name] = widgets.HTML('')
    _kc_usd_labels[kc.name] = widgets.HTML('')

_kc_merge_inputs = {}
_kc_cards_inputs = {}
for kc in _kc_tiers:
    _kc_merge_inputs[kc.name] = widgets.FloatText(value=kc.merge_cost_nuts, step=10, layout=_w)
    _kc_cards_inputs[kc.name] = widgets.IntText(value=kc.cards_required, step=1, layout=_w)

_kc_merge_inputs[_kc_names[0]].disabled = True
_kc_cards_inputs[_kc_names[0]].disabled = True

# ── Instance Loot inputs (per-tier) ───────────────────────────────────────
_loot_nuts = {}
_loot_scrap = {}
_loot_coins = {}
_loot_xp = {}
_loot_gear = {}
_loot_kc_drop = {}
_lw = widgets.Layout(width='100px')
for tier in econ_params.instance_tiers:
    _loot_nuts[tier.name] = widgets.FloatText(value=tier.nuts_earned, step=5, layout=_lw)
    _loot_scrap[tier.name] = widgets.FloatText(value=tier.scrap_earned, step=10, layout=_lw)
    _loot_coins[tier.name] = widgets.FloatText(value=tier.coins_earned, step=0.5, layout=_lw)
    _loot_xp[tier.name] = widgets.FloatText(value=tier.xp_earned, step=10, layout=_lw)
    _loot_gear[tier.name] = widgets.FloatText(value=tier.gear_value_usd, step=0.05, layout=_lw)
    _loot_kc_drop[tier.name] = widgets.FloatText(value=tier.keycard_drop_chance * 100, step=1, layout=_lw)

# ── Economy sliders ───────────────────────────────────────────────────────
runs_slider = widgets.IntSlider(value=econ_params.instances_per_day, min=1, max=10, step=1,
                                 description='Runs/day:')
buff_cost_slider = widgets.FloatSlider(value=econ_params.buff_cost_scrap, min=0, max=200, step=10,
                                       description='Buff cost:')
bp_rate_slider = widgets.FloatSlider(value=econ_params.battle_pass.purchase_rate * 100,
                                      min=1, max=30, step=1, description='BP buyers %:')

# ── Battle Pass inputs ────────────────────────────────────────────────────
bp_xp_to_complete = widgets.FloatText(value=econ_params.battle_pass.xp_to_complete, step=1000,
                                       description='BP XP:', layout=widgets.Layout(width='200px'))
bp_gear_count = widgets.IntText(value=econ_params.battle_pass.gear_reward_count, step=1,
                                 description='Gear items:', layout=widgets.Layout(width='200px'))
bp_gear_value = widgets.FloatText(value=econ_params.battle_pass.gear_avg_value_usd, step=0.5,
                                   description='Avg gear $:', layout=widgets.Layout(width='200px'))
bp_season_days_input = widgets.IntText(value=econ_params.battle_pass.season_days, step=10,
                                        description='Season days:', layout=widgets.Layout(width='200px'))

_anchors = state.retention_anchors if state else cfg.retention.anchors
_monetization = state.monetization if state else cfg.monetization

def _make_econ_params(runs, buff_cost, bp_rate):
    from aco_model.models import KeyCardTier, InstanceTier, BattlePassParams
    updated_kc = []
    for kc in _kc_tiers:
        updated_kc.append(KeyCardTier(
            name=kc.name,
            cards_required=_kc_cards_inputs[kc.name].value,
            merge_cost_nuts=_kc_merge_inputs[kc.name].value,
            instance_tier=kc.instance_tier,
        ))
    updated_tiers = []
    for tier in econ_params.instance_tiers:
        updated_tiers.append(InstanceTier(
            name=tier.name,
            nuts_earned=_loot_nuts[tier.name].value,
            scrap_earned=_loot_scrap[tier.name].value,
            coins_earned=_loot_coins[tier.name].value,
            xp_earned=_loot_xp[tier.name].value,
            gear_value_usd=_loot_gear[tier.name].value,
            keycard_drop_chance=_loot_kc_drop[tier.name].value / 100.0,
        ))
    updated_bp = econ_params.battle_pass.model_copy(update={
        'purchase_rate': bp_rate / 100.0,
        'xp_to_complete': bp_xp_to_complete.value,
        'gear_reward_count': bp_gear_count.value,
        'gear_avg_value_usd': bp_gear_value.value,
        'season_days': bp_season_days_input.value,
    })
    return econ_params.model_copy(update={
        'instances_per_day': runs, 'buff_cost_scrap': buff_cost,
        'battle_pass': updated_bp, 'keycard_tiers': updated_kc,
        'instance_tiers': updated_tiers, 'coin_to_usd': coin_value.value,
        'seed_coins': seed_coins.value,
        'seed_nuts': seed_nuts.value,
        'seed_scrap': seed_scrap.value,
    })

def _get_keycard_coin_costs():
    """Derive coin costs and total USD value (including merge costs) for each tier."""
    coin_costs = {}
    total_usd = {}
    cv = coin_value.value
    nv = nut_value.value
    base = bronze_coin_cost.value
    coin_costs[_kc_names[0]] = base
    total_usd[_kc_names[0]] = base * cv
    for i in range(1, len(_kc_tiers)):
        name = _kc_names[i]
        cards_needed = _kc_cards_inputs[name].value
        merge_nuts = _kc_merge_inputs[name].value
        prev_name = _kc_names[i - 1]
        prev_coin = coin_costs[prev_name]
        coin_costs[name] = prev_coin * cards_needed if cards_needed > 0 else prev_coin
        prev_total = total_usd[prev_name]
        total_usd[name] = (cards_needed * prev_total) + (merge_nuts * nv)
    return coin_costs, total_usd

# ── Defaults for reset/save ──────────────────────────────────────────────
_econ_defaults = {
    'coin_value': coin_value.value,
    'nut_value': nut_value.value,
    'scrap_value': scrap_value.value,
    'seed_coins': seed_coins.value,
    'seed_nuts': seed_nuts.value,
    'seed_scrap': seed_scrap.value,
    'runs_per_day': runs_slider.value,
    'buff_cost': buff_cost_slider.value,
    'bp_rate': bp_rate_slider.value,
    'bronze_coin_cost': bronze_coin_cost.value,
    'bp_xp_to_complete': bp_xp_to_complete.value,
    'bp_gear_count': bp_gear_count.value,
    'bp_gear_value': bp_gear_value.value,
    'bp_season_days': bp_season_days_input.value,
    'merge_costs': {kc.name: _kc_merge_inputs[kc.name].value for kc in _kc_tiers},
    'cards_required': {kc.name: _kc_cards_inputs[kc.name].value for kc in _kc_tiers},
    'loot_nuts': {t.name: _loot_nuts[t.name].value for t in econ_params.instance_tiers},
    'loot_scrap': {t.name: _loot_scrap[t.name].value for t in econ_params.instance_tiers},
    'loot_coins': {t.name: _loot_coins[t.name].value for t in econ_params.instance_tiers},
    'loot_xp': {t.name: _loot_xp[t.name].value for t in econ_params.instance_tiers},
    'loot_gear': {t.name: _loot_gear[t.name].value for t in econ_params.instance_tiers},
    'loot_kc_drop': {t.name: _loot_kc_drop[t.name].value for t in econ_params.instance_tiers},
}

def _reset_to_defaults(_):
    coin_value.value = _econ_defaults['coin_value']
    nut_value.value = _econ_defaults['nut_value']
    scrap_value.value = _econ_defaults['scrap_value']
    seed_coins.value = _econ_defaults['seed_coins']
    seed_nuts.value = _econ_defaults['seed_nuts']
    seed_scrap.value = _econ_defaults['seed_scrap']
    runs_slider.value = _econ_defaults['runs_per_day']
    buff_cost_slider.value = _econ_defaults['buff_cost']
    bp_rate_slider.value = _econ_defaults['bp_rate']
    bronze_coin_cost.value = _econ_defaults['bronze_coin_cost']
    bp_xp_to_complete.value = _econ_defaults['bp_xp_to_complete']
    bp_gear_count.value = _econ_defaults['bp_gear_count']
    bp_gear_value.value = _econ_defaults['bp_gear_value']
    bp_season_days_input.value = _econ_defaults['bp_season_days']
    for kc in _kc_tiers:
        _kc_merge_inputs[kc.name].value = _econ_defaults['merge_costs'][kc.name]
        _kc_cards_inputs[kc.name].value = _econ_defaults['cards_required'][kc.name]
    for t in econ_params.instance_tiers:
        _loot_nuts[t.name].value = _econ_defaults['loot_nuts'][t.name]
        _loot_scrap[t.name].value = _econ_defaults['loot_scrap'][t.name]
        _loot_coins[t.name].value = _econ_defaults['loot_coins'][t.name]
        _loot_xp[t.name].value = _econ_defaults['loot_xp'][t.name]
        _loot_gear[t.name].value = _econ_defaults['loot_gear'][t.name]
        _loot_kc_drop[t.name].value = _econ_defaults['loot_kc_drop'][t.name]
    save_status.value = 'Reset to defaults'

def _save_as_defaults(_):
    import yaml
    _econ_defaults['coin_value'] = coin_value.value
    _econ_defaults['nut_value'] = nut_value.value
    _econ_defaults['scrap_value'] = scrap_value.value
    _econ_defaults['seed_coins'] = seed_coins.value
    _econ_defaults['seed_nuts'] = seed_nuts.value
    _econ_defaults['seed_scrap'] = seed_scrap.value
    _econ_defaults['runs_per_day'] = runs_slider.value
    _econ_defaults['buff_cost'] = buff_cost_slider.value
    _econ_defaults['bp_rate'] = bp_rate_slider.value
    _econ_defaults['bronze_coin_cost'] = bronze_coin_cost.value
    _econ_defaults['bp_xp_to_complete'] = bp_xp_to_complete.value
    _econ_defaults['bp_gear_count'] = bp_gear_count.value
    _econ_defaults['bp_gear_value'] = bp_gear_value.value
    _econ_defaults['bp_season_days'] = bp_season_days_input.value
    for kc in _kc_tiers:
        _econ_defaults['merge_costs'][kc.name] = _kc_merge_inputs[kc.name].value
        _econ_defaults['cards_required'][kc.name] = _kc_cards_inputs[kc.name].value
    for t in econ_params.instance_tiers:
        _econ_defaults['loot_nuts'][t.name] = _loot_nuts[t.name].value
        _econ_defaults['loot_scrap'][t.name] = _loot_scrap[t.name].value
        _econ_defaults['loot_coins'][t.name] = _loot_coins[t.name].value
        _econ_defaults['loot_xp'][t.name] = _loot_xp[t.name].value
        _econ_defaults['loot_gear'][t.name] = _loot_gear[t.name].value
        _econ_defaults['loot_kc_drop'][t.name] = _loot_kc_drop[t.name].value
    params = _make_econ_params(runs_slider.value, buff_cost_slider.value, bp_rate_slider.value)
    config_path = Path('config.yaml')
    with open(config_path) as f:
        data = yaml.safe_load(f) or {}
    data['economy'] = yaml.safe_load(params.model_dump_json())
    data['economy']['coin_to_usd'] = coin_value.value
    data['economy']['bronze_coin_cost'] = bronze_coin_cost.value
    data['economy']['nut_value_usd'] = nut_value.value
    data['economy']['scrap_value_usd'] = scrap_value.value
    with open(config_path, 'w') as f:
        yaml.dump(data, f, default_flow_style=None, sort_keys=False)
    save_status.value = f'Saved to config.yaml'

reset_btn = widgets.Button(description='Reset to Defaults', button_style='warning',
                           layout=widgets.Layout(width='160px'))
reset_btn.on_click(_reset_to_defaults)
save_btn = widgets.Button(description='Save as Defaults', button_style='success',
                          layout=widgets.Layout(width='160px'))
save_btn.on_click(_save_as_defaults)
save_status = widgets.Label(value='')

# ── Output widgets ────────────────────────────────────────────────────────
out_rates = widgets.Output()
out_ie = widgets.Output()
out_player = widgets.Output()
out_flows = widgets.Output()
out_kc = widgets.Output()
out_bp = widgets.Output()

def _update_all(*args):
    runs = runs_slider.value
    buff_cost = buff_cost_slider.value
    bp_rate = bp_rate_slider.value
    nv = nut_value.value
    sv = scrap_value.value
    cv = coin_value.value
    kc_costs, kc_total_usd = _get_keycard_coin_costs()

    params = _make_econ_params(runs, buff_cost, bp_rate)
    econ = simulate_economy(sim, params)
    days = np.arange(1, sim.sim_days + 1)

    # Update derived price labels
    for name, cost in kc_costs.items():
        _kc_derived_prices[name].value = f'<div style="width:120px;line-height:28px;font-family:monospace">{cost:,.2f}</div>'
        _kc_usd_labels[name].value = f'<div style="width:100px;line-height:28px;font-family:monospace">${kc_total_usd[name]:,.2f}</div>'

    # ── Exchange Rate & Value Tables ──────────────────────────────────
    with out_rates:
        out_rates.clear_output(wait=True)
        base_kc_usd = kc_costs[_kc_names[0]] * cv
        rates_data = {
            'Resource': ['1 Coin', '1 Nut', '1 Scrap', '1 Keycard (bronze)'],
            'USD': [f'${cv:.2f}', f'${nv:.4f}', f'${sv:.4f}', f'${base_kc_usd:.2f}'],
            'Coins': ['1.00',
                f'{nv / cv:.4f}' if cv > 0 else '-',
                f'{sv / cv:.4f}' if cv > 0 else '-',
                f'{base_kc_usd / cv:.2f}' if cv > 0 else '-'],
            'Nuts': [f'{cv / nv:,.0f}' if nv > 0 else '-', '1.00',
                f'{sv / nv:.2f}' if nv > 0 else '-',
                f'{base_kc_usd / nv:,.0f}' if nv > 0 else '-'],
            'Scrap': [f'{cv / sv:,.0f}' if sv > 0 else '-',
                f'{nv / sv:.2f}' if sv > 0 else '-', '1.00',
                f'{base_kc_usd / sv:,.0f}' if sv > 0 else '-'],
        }
        print('=== Exchange Rates ===')
        print(pd.DataFrame(rates_data).to_string(index=False))

        print('\n=== Seed Currency (new player starting wallet) ===')
        print(f'  {seed_coins.value:,.0f} coins (${seed_coins.value * cv:,.2f})')
        print(f'  {seed_nuts.value:,.0f} nuts (${seed_nuts.value * nv:,.2f})')
        print(f'  {seed_scrap.value:,.0f} scrap (${seed_scrap.value * sv:,.2f})')
        total_seed_usd = seed_coins.value * cv + seed_nuts.value * nv + seed_scrap.value * sv
        print(f'  Total seed value: ${total_seed_usd:,.2f}')

        print('\n=== Key Card Costs ===')
        kc_rows = []
        for i, kc in enumerate(params.keycard_tiers):
            coin_cost = kc_costs[kc.name]
            merge_nuts = kc.merge_cost_nuts
            merge_usd = merge_nuts * nv
            total_val = kc_total_usd[kc.name]
            prev_tier = _kc_names[i - 1] if i > 0 else '-'
            cards_label = f'{kc.cards_required} {prev_tier}' if kc.cards_required > 0 else '-'
            kc_rows.append({
                'Keycard': kc.name,
                'Buy (coins)': f'{coin_cost:,.2f}',
                'Total Value': f'${total_val:,.2f}',
                'Cards to Merge': cards_label,
                'Merge (nuts)': f'{merge_nuts:.0f}' if merge_nuts > 0 else '-',
                'Merge (USD)': f'${merge_usd:.2f}' if merge_nuts > 0 else '-',
                'Unlocks': kc.instance_tier,
            })
        print(pd.DataFrame(kc_rows).to_string(index=False))

        print('\n=== Instance Value per Run (USD) ===')
        rows = []
        for i, tier in enumerate(params.instance_tiers):
            kct = params.keycard_tiers[i] if i < len(params.keycard_tiers) else None
            kc_usd = kc_total_usd[kct.name] if kct else 0
            buff_usd = buff_cost * sv * params.buffs_per_run
            merge_usd = 0
            if kct and kct.merge_cost_nuts > 0 and kct.cards_required > 0:
                merge_usd = (kct.merge_cost_nuts * nv) / kct.cards_required
            total_in = buff_usd + kc_usd + merge_usd
            nuts_out_usd = tier.nuts_earned * nv
            scrap_out_usd = tier.scrap_earned * sv
            coins_out_usd = tier.coins_earned * cv
            gear_out_usd = tier.gear_value_usd
            keycard_back_usd = tier.keycard_drop_chance * kc_total_usd[kct.name] if kct else 0
            total_out = nuts_out_usd + scrap_out_usd + coins_out_usd + gear_out_usd + keycard_back_usd
            rows.append({
                'Tier': tier.name, 'KC In': f'${kc_usd:.2f}',
                'Buff': f'${buff_usd:.2f}', 'Merge': f'${merge_usd:.2f}',
                'Total In': f'${total_in:.2f}', 'Nuts': f'${nuts_out_usd:.2f}',
                'Scrap': f'${scrap_out_usd:.2f}', 'Coins': f'${coins_out_usd:.2f}',
                'Gear': f'${gear_out_usd:.2f}', 'KC Back': f'${keycard_back_usd:.2f}',
                'Total Out': f'${total_out:.2f}',
                'Net': f'${total_out - total_in:+.2f}',
            })
        print(pd.DataFrame(rows).to_string(index=False))
        avg_net = np.mean([float(r['Net'].replace('$','').replace('+','')) for r in rows])
        print(f'\nAvg net value per run: ${avg_net:+.2f} | Per player per day ({runs} runs): ${avg_net * runs:+.2f}')

    # ── Instance Economics (USD charts) ───────────────────────────────
    with out_ie:
        out_ie.clear_output(wait=True)
        tier_names = [r['Tier'] for r in rows]
        total_in_vals = [float(r['Total In'].replace('$','')) for r in rows]
        total_out_vals = [float(r['Total Out'].replace('$','')) for r in rows]
        net_vals = [float(r['Net'].replace('$','').replace('+','')) for r in rows]

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        x = np.arange(len(tier_names)); width = 0.35
        ax1.bar(x - width/2, total_in_vals, width, label='Value In', color='#F44336', alpha=0.7)
        ax1.bar(x + width/2, total_out_vals, width, label='Value Out', color='#4CAF50', alpha=0.7)
        ax1.set_xticks(x); ax1.set_xticklabels(tier_names)
        ax1.set_ylabel('USD'); ax1.set_title('Value In vs Out per Instance Run (USD)'); ax1.legend()
        colors = ['#4CAF50' if v >= 0 else '#F44336' for v in net_vals]
        ax2.bar(tier_names, net_vals, color=colors, alpha=0.7)
        ax2.set_ylabel('USD'); ax2.set_title('Net Value per Instance Run (USD)')
        ax2.axhline(y=0, color='black', linestyle='-', alpha=0.3)
        plt.tight_layout(); plt.show()

    # ── Section 2: Player Wallet Progression ─────────────────────────
    player_days = max(sim.sim_days, 180)
    df_no_bp = simulate_player_progression(params, max_player_days=player_days, has_battle_pass=False)
    df_bp = simulate_player_progression(params, max_player_days=player_days, has_battle_pass=True)

    with out_player:
        out_player.clear_output(wait=True)

        # Graphs by player day
        fig, axes = plt.subplots(2, 4, figsize=(20, 9))
        for ax_row, df, label in [(axes[0], df_no_bp, 'Non-Payer (no BP)'), (axes[1], df_bp, 'BP Holder')]:
            # Aggregate to player day (take last value of each day)
            daily = df.groupby('player_day').last().reset_index()
            for ax, col, color, title in [
                (ax_row[0], 'coins', '#FFD700', 'Coins'),
                (ax_row[1], 'nuts', '#FF9800', 'Nuts'),
                (ax_row[2], 'scrap', '#9C27B0', 'Scrap'),
                (ax_row[3], 'xp', '#2196F3', 'XP'),
            ]:
                ax.plot(daily['player_day'], daily[col], linewidth=2, color=color)
                ax.fill_between(daily['player_day'], daily[col], alpha=0.2, color=color)
                # Mark tier-ups
                tier_ups = df[df['tier_up']].groupby('player_day').first().reset_index()
                for _, tu in tier_ups.iterrows():
                    ax.axvline(x=tu['player_day'], color='red', linestyle='--', alpha=0.3, linewidth=0.5)
                ax.set_xlabel('Player Day')
                ax.set_ylabel(title)
                ax.set_title(f'{label}: {title}')
        plt.tight_layout(); plt.show()

        # Summary
        print('=== Non-Payer Summary ===')
        final_no_bp = df_no_bp.iloc[-1]
        print(f'  Final tier: {final_no_bp["instance_tier"]} (keycard: {final_no_bp["keycard_tier"]})')
        print(f'  Final wallet: {final_no_bp["coins"]:,.0f} coins, {final_no_bp["nuts"]:,.0f} nuts, {final_no_bp["scrap"]:,.0f} scrap')
        print(f'  Total XP: {final_no_bp["xp"]:,.0f}')
        tier_ups_no_bp = df_no_bp[df_no_bp["tier_up"]]
        print(f'  Tier-ups: {len(tier_ups_no_bp)} ({list(tier_ups_no_bp["keycard_tier"].tolist())})')

        print('\n=== BP Holder Summary ===')
        final_bp = df_bp.iloc[-1]
        print(f'  Final tier: {final_bp["instance_tier"]} (keycard: {final_bp["keycard_tier"]})')
        print(f'  Final wallet: {final_bp["coins"]:,.0f} coins, {final_bp["nuts"]:,.0f} nuts, {final_bp["scrap"]:,.0f} scrap')
        print(f'  Total XP: {final_bp["xp"]:,.0f}')
        tier_ups_bp = df_bp[df_bp["tier_up"]]
        print(f'  Tier-ups: {len(tier_ups_bp)} ({list(tier_ups_bp["keycard_tier"].tolist())})')

    # ── Section 3: Total Economy Balance ──────────────────────────────
    with out_flows:
        out_flows.clear_output(wait=True)
        # 2x3: rows = earned/spent, net; cols = coins, nuts, scrap
        fig, axes = plt.subplots(2, 3, figsize=(18, 9))

        # Derive coins earned/spent
        coins_earned_daily = econ.daily_coins_earned + econ.daily_coins_from_bp
        coins_spent_daily = econ.daily_coins_from_bp  # simplified: players spending on BP
        coins_net = np.cumsum(coins_earned_daily - coins_spent_daily)
        nuts_net = econ.nuts_balance
        scrap_net = econ.scrap_balance

        # Row 1: Daily earned vs spent
        for ax, earned, spent, name, color_e, color_s in [
            (axes[0, 0], coins_earned_daily, coins_spent_daily, 'Coins', '#FFD700', '#F44336'),
            (axes[0, 1], econ.daily_nuts_earned, econ.daily_nuts_spent, 'Nuts', '#FF9800', '#F44336'),
            (axes[0, 2], econ.daily_scrap_earned, econ.daily_scrap_spent, 'Scrap', '#9C27B0', '#F44336'),
        ]:
            ax.fill_between(days, earned, alpha=0.3, color='#4CAF50', label='Earned')
            ax.fill_between(days, spent, alpha=0.3, color=color_s, label='Spent')
            ax.plot(days, earned, linewidth=2, color='#4CAF50')
            ax.plot(days, spent, linewidth=2, color=color_s)
            ax.set_xlabel('Sim Day'); ax.set_ylabel(name)
            ax.set_title(f'Daily {name}: Earned vs Spent'); ax.legend()

        # Row 2: Cumulative net balance
        for ax, net, name, color in [
            (axes[1, 0], coins_net, 'Coins', '#FFD700'),
            (axes[1, 1], nuts_net, 'Nuts', '#FF9800'),
            (axes[1, 2], scrap_net, 'Scrap', '#9C27B0'),
        ]:
            ax.plot(days, net, linewidth=2, color=color)
            ax.fill_between(days, net, alpha=0.2, color=color)
            ax.set_xlabel('Sim Day'); ax.set_ylabel(name)
            ax.set_title(f'Cumulative {name} Balance in System')

        plt.tight_layout(); plt.show()

        # Summary
        print('=== Total Economy Balance (over simulation) ===')
        print(f'  Coins: earned {coins_earned_daily.sum():,.0f}, spent {coins_spent_daily.sum():,.0f}, balance {coins_net[-1]:,.0f}')
        print(f'  Nuts:  earned {econ.daily_nuts_earned.sum():,.0f}, spent {econ.daily_nuts_spent.sum():,.0f}, balance {nuts_net[-1]:,.0f}')
        print(f'  Scrap: earned {econ.daily_scrap_earned.sum():,.0f}, spent {econ.daily_scrap_spent.sum():,.0f}, balance {scrap_net[-1]:,.0f}')

    # ── Section 4: Key Card Progression ──────────────────────────────
    with out_kc:
        out_kc.clear_output(wait=True)
        # Use player progression to find runs-to-afford each tier
        # Find first run where player reaches each keycard tier
        tier_reach = {}
        for kc_name in _kc_names:
            mask = df_no_bp['keycard_tier'] == kc_name
            if mask.any():
                first = df_no_bp[mask].iloc[0]
                tier_reach[kc_name] = {
                    'run': first['run'],
                    'player_day': first['player_day']
                }

        # Chart: runs to reach each tier
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        reached_names = list(tier_reach.keys())
        runs_vals = [tier_reach[n]['run'] for n in reached_names]
        days_vals = [tier_reach[n]['player_day'] for n in reached_names]

        ax1.bar(reached_names, runs_vals, color='#2196F3', alpha=0.7)
        ax1.set_ylabel('Runs'); ax1.set_title('Runs to Reach Each Keycard Tier (Non-Payer)')
        for i, v in enumerate(runs_vals):
            ax1.text(i, v, f'{v}', ha='center', va='bottom')

        ax2.bar(reached_names, days_vals, color='#FF9800', alpha=0.7)
        ax2.set_ylabel('Player Days'); ax2.set_title('Days to Reach Each Keycard Tier (Non-Payer)')
        for i, v in enumerate(days_vals):
            ax2.text(i, v, f'{v}', ha='center', va='bottom')
        plt.tight_layout(); plt.show()

        print('=== Key Card Progression (Non-Payer) ===')
        for name, info in tier_reach.items():
            print(f'  {name}: reached at run {info["run"]}, player day {info["player_day"]}')

        unreached = [n for n in _kc_names if n not in tier_reach]
        if unreached:
            print(f'\n  Not reached: {unreached}')

    # ── Section 5: Battle Pass Economics ─────────────────────────────
    with out_bp:
        out_bp.clear_output(wait=True)
        bp = params.battle_pass
        season_days_val = bp.season_days
        runs_in_season = runs * season_days_val
        avg_xp_per_run = np.mean([t.xp_earned for t in params.instance_tiers])
        total_xp_season = runs_in_season * avg_xp_per_run

        # Player profiles
        profiles = [
            ('Low (1 run/day)', 1),
            ('Medium (3 runs/day)', 3),
            ('High (5 runs/day)', 5),
        ]

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

        # Chart 1: XP earned vs BP XP target by profile
        profile_names = [p[0] for p in profiles]
        xp_totals = [p[1] * season_days_val * avg_xp_per_run for p in profiles]
        bars = ax1.bar(profile_names, xp_totals, color='#2196F3', alpha=0.7, label='XP earned in season')
        ax1.axhline(y=bp.xp_to_complete, color='red', linestyle='--', linewidth=2, label=f'BP XP target ({bp.xp_to_complete:,.0f})')
        ax1.set_ylabel('Total XP'); ax1.set_title('XP Earned vs BP Target by Player Profile')
        ax1.legend()
        for i, v in enumerate(xp_totals):
            ax1.text(i, v, f'{v:,.0f}', ha='center', va='bottom')

        # Chart 2: Days to complete BP by profile
        days_to_complete = []
        for _, runs_per_day_profile in profiles:
            xp_per_day = runs_per_day_profile * avg_xp_per_run
            if xp_per_day > 0:
                d = bp.xp_to_complete / xp_per_day
            else:
                d = float('inf')
            days_to_complete.append(d)

        colors_bar = ['#4CAF50' if d <= season_days_val else '#F44336' for d in days_to_complete]
        ax2.bar(profile_names, days_to_complete, color=colors_bar, alpha=0.7)
        ax2.axhline(y=season_days_val, color='black', linestyle='--', linewidth=1, label=f'Season length ({season_days_val}d)')
        ax2.set_ylabel('Days'); ax2.set_title('Days to Complete BP by Player Profile')
        ax2.legend()
        for i, v in enumerate(days_to_complete):
            label = f'{v:.0f}d' if v != float('inf') else 'never'
            ax2.text(i, v if v != float('inf') else season_days_val, label, ha='center', va='bottom')
        plt.tight_layout(); plt.show()

        # Total BP payout and net
        bp_nuts_usd = bp.nuts_reward_total * nv
        bp_scrap_usd = bp.scrap_reward_total * sv
        bp_kc_usd = bp.keycards_rewarded * kc_costs[_kc_names[0]] * cv
        bp_coins_usd = bp.coins_returned * cv
        bp_gear_usd = bp.gear_reward_count * bp.gear_avg_value_usd
        bp_total_value = bp_nuts_usd + bp_scrap_usd + bp_kc_usd + bp_coins_usd + bp_gear_usd
        bp_cost_usd = bp.cost_coins * cv

        # Per-XP reward rate
        value_per_xp = bp_total_value / bp.xp_to_complete if bp.xp_to_complete > 0 else 0
        per_run_boost = value_per_xp * avg_xp_per_run

        print(f'=== Battle Pass Economics ===')
        print(f'  Cost: {bp.cost_coins} coins (${bp_cost_usd:.2f})')
        print(f'  Season: {season_days_val} days')
        print(f'  XP to complete: {bp.xp_to_complete:,.0f}')
        print(f'')
        print(f'=== BP Reward Payout (if completed) ===')
        print(f'  Coins back: {bp.coins_returned} (${bp_coins_usd:.2f})')
        print(f'  Nuts: {bp.nuts_reward_total:,.0f} (${bp_nuts_usd:.2f})')
        print(f'  Scrap: {bp.scrap_reward_total:,.0f} (${bp_scrap_usd:.2f})')
        print(f'  Keycards: {bp.keycards_rewarded} bronze (${bp_kc_usd:.2f})')
        print(f'  Gear: {bp.gear_reward_count} items @ ${bp.gear_avg_value_usd:.2f} (${bp_gear_usd:.2f})')
        print(f'  Total reward value: ${bp_total_value:.2f}')
        print(f'  Net to player: ${bp_total_value - bp_cost_usd:+.2f}')
        print(f'  Player ROI: {bp_total_value / bp_cost_usd:.1f}x' if bp_cost_usd > 0 else '')
        print(f'')
        print(f'=== Per-Run BP Value Boost ===')
        print(f'  Value per XP point: ${value_per_xp:.4f}')
        print(f'  Avg XP per run: {avg_xp_per_run:.0f}')
        print(f'  Extra value per run (BP holder): ${per_run_boost:.2f}')
        print(f'')
        print(f'  Total BP revenue (all DAU): ${econ.battle_pass_total_revenue:,.2f}')

    save_state(sim, _anchors, monetization=_monetization, economy=params)

# ── Build keycard input grid ──────────────────────────────────────────────
_kc_rows_ui = []
_kc_rows_ui.append(widgets.HBox([
    widgets.HTML('<div style="width:90px;line-height:28px;font-weight:bold">bronze</div>'),
    bronze_coin_cost,
    widgets.HTML('<div style="width:120px;line-height:28px;color:#999">(base tier)</div>'),
    widgets.HTML('<div style="width:120px"></div>'),
    widgets.HTML('<div style="width:100px"></div>'),
    _kc_usd_labels[_kc_names[0]],
]))
for i in range(1, len(_kc_tiers)):
    kc = _kc_tiers[i]
    prev_name = _kc_names[i - 1]
    _kc_rows_ui.append(widgets.HBox([
        widgets.HTML(f'<div style="width:90px;line-height:28px;font-weight:bold">{kc.name}</div>'),
        _kc_derived_prices[kc.name],
        _kc_merge_inputs[kc.name],
        _kc_cards_inputs[kc.name],
        widgets.HTML(f'<div style="width:100px;line-height:28px">{prev_name} cards</div>'),
        _kc_usd_labels[kc.name],
    ]))

_kc_header_row = widgets.HBox([
    widgets.HTML('<div style="width:90px;font-weight:bold">Tier</div>'),
    widgets.HTML('<div style="width:120px;font-weight:bold">Buy Price (coins)</div>'),
    widgets.HTML('<div style="width:120px;font-weight:bold">Merge Cost (nuts)</div>'),
    widgets.HTML('<div style="width:220px;font-weight:bold">Cards Required</div>'),
    widgets.HTML('<div style="width:100px;font-weight:bold">$ Value</div>'),
])

# ── Build loot input grid ─────────────────────────────────────────────────
_loot_header = widgets.HBox([
    widgets.HTML('<div style="width:90px;font-weight:bold">Tier</div>'),
    widgets.HTML('<div style="width:100px;font-weight:bold">Nuts</div>'),
    widgets.HTML('<div style="width:100px;font-weight:bold">Scrap</div>'),
    widgets.HTML('<div style="width:100px;font-weight:bold">Coins</div>'),
    widgets.HTML('<div style="width:100px;font-weight:bold">XP</div>'),
    widgets.HTML('<div style="width:100px;font-weight:bold">Gear ($)</div>'),
    widgets.HTML('<div style="width:100px;font-weight:bold">KC Drop %</div>'),
])
_loot_rows_ui = []
for tier in econ_params.instance_tiers:
    _loot_rows_ui.append(widgets.HBox([
        widgets.HTML(f'<div style="width:90px;line-height:28px;font-weight:bold">{tier.name}</div>'),
        _loot_nuts[tier.name],
        _loot_scrap[tier.name],
        _loot_coins[tier.name],
        _loot_xp[tier.name],
        _loot_gear[tier.name],
        _loot_kc_drop[tier.name],
    ]))

# ── Collect all observable widgets ────────────────────────────────────────
_all_inputs = [runs_slider, buff_cost_slider, bp_rate_slider, coin_value, nut_value, scrap_value,
               seed_coins, seed_nuts, seed_scrap, bronze_coin_cost,
               bp_xp_to_complete, bp_gear_count, bp_gear_value, bp_season_days_input]
for kc in _kc_tiers:
    _all_inputs.extend([_kc_merge_inputs[kc.name], _kc_cards_inputs[kc.name]])
for t in econ_params.instance_tiers:
    _all_inputs.extend([_loot_nuts[t.name], _loot_scrap[t.name], _loot_coins[t.name],
                        _loot_xp[t.name], _loot_gear[t.name], _loot_kc_drop[t.name]])

if _HEADLESS:
    _update_all()
else:
    for s in _all_inputs:
        s.observe(lambda change: _update_all(), names='value')

    display(widgets.VBox([
        widgets.HBox([coin_value, nut_value, scrap_value]),
        widgets.HBox([seed_coins, seed_nuts, seed_scrap]),
        widgets.HBox([runs_slider, buff_cost_slider, bp_rate_slider]),
        widgets.HBox([bp_xp_to_complete, bp_gear_count, bp_gear_value, bp_season_days_input]),
        widgets.HBox([reset_btn, save_btn, save_status]),
        widgets.HTML('<hr><b>Key Card Costs</b>'),
        _kc_header_row,
        *_kc_rows_ui,
        widgets.HTML('<hr><b>Instance Loot (avg per run)</b>'),
        _loot_header,
        *_loot_rows_ui,
        widgets.HTML('<hr>'),
        out_rates,
        out_ie,
    ]))
    _update_all()


## 2. Player Wallet Progression

Average player wallet balances after install. Shows a single player's progression through tiers.

In [ ]:
if not os.environ.get('ACO_HEADLESS'):
    display(out_player)

## 3. Total Economy Balance

Total amount of currency in the system over the simulation.

In [ ]:
if not os.environ.get('ACO_HEADLESS'):
    display(out_flows)

## 4. Key Card Progression

In [ ]:
if not os.environ.get('ACO_HEADLESS'):
    display(out_kc)

## 5. Battle Pass Economics

In [ ]:
if not os.environ.get('ACO_HEADLESS'):
    display(out_bp)